# 🧪 A/B 测试从 0 到 1：电商推荐算法实验

> **商业背景**: 某电商平台上线了新的推荐算法，希望评估该算法是否能提升用户的 **购买转化率 (Purchase Conversion Rate)**。
> 
> **学习目标**: 走完一个完整 A/B 实验的全流程 SOP：假设定义 → 样本量计算 → SRM 检测 → 显著性检验 → 分层分析 → 业务决策。
>
> **数据**: Kaggle `mkechinov/ecommerce-behavior-data-from-multi-category-store`

---

## 0. 函数加油站 (Function Cheat Sheet) 🛠️

| 函数 | 大白话 | 标准语法 | SQL 类比 |
| :--- | :--- | :--- | :--- |
| `stats.ttest_ind(a, b)` | 两组均值有没有差异？ | `stat, p = stats.ttest_ind(group_a, group_b)` | `SELECT AVG(metric) FROM groups GROUP BY variant` |
| `stats.chisquare(obs, exp)` | 实际分布和期望分布一样吗？(SRM) | `stat, p = stats.chisquare([n_ctrl, n_test], [exp, exp])` | `SELECT COUNT(*) FROM users GROUP BY variant` |
| `stats.chi2_contingency(tbl)` | 两组的转化率有差异吗？(频数) | `chi2, p, dof, _ = stats.chi2_contingency(table)` | `SELECT variant, converted, COUNT(*) ... GROUP BY 1, 2` |
| `sms.proportion_effectsize(p1, p2)` | 效应量有多大？(Cohen's h) | `es = sms.proportion_effectsize(0.10, 0.11)` | N/A (纯统计计算) |
| `NormalIndPower().solve_power(...)` | 需要多少样本？ | `n = NormalIndPower().solve_power(es, power=0.8, alpha=0.05)` | N/A (实验设计阶段) |
| `sms.CompareMeans(...)` | 两组差异的置信区间 | `cm.tconfint_diff(usevar='unequal')` | N/A (统计推断) |

## 1. 商业背景与假设定义 📋

### 1.1 业务场景

某电商平台（类似淘宝/京东）上线了 **新推荐算法 v2**，产品经理想知道：

> 💡 "新推荐算法是否真的能提升用户的购买转化率？"

### 1.2 指标定义

| 指标类型 | 指标名称 | 定义 | 公式 |
| :--- | :--- | :--- | :--- |
| **主指标** | 购买转化率 (CVR) | 有购买行为的用户占总访问用户的比例 | `购买用户数 / 总用户数` |
| **护栏指标** | 人均浏览量 | 用户平均浏览商品数 | `总浏览事件数 / 总用户数` |
| **护栏指标** | 加购率 | 有加购行为的用户占比 | `加购用户数 / 总用户数` |

### 1.3 假设定义

- **H₀ (零假设)**: 新推荐算法的购买转化率 = 旧算法的购买转化率 (CVR_test = CVR_ctrl)
- **H₁ (备择假设)**: 新推荐算法的购买转化率 ≠ 旧算法的购买转化率 (CVR_test ≠ CVR_ctrl)
- **显著性水平**: α = 0.05
- **统计功效**: Power = 0.80

## 2. 数据准备 📦

### 2.1 数据下载与加载

In [2]:
# 数据导入 (Kaggle API)
# 如果 kaggle 不在 PATH，使用完整路径：
# kaggle

!kaggle datasets download mkechinov/ecommerce-behavior-data-from-multi-category-store \
    --path ./data --unzip -f 2019-Oct.csv

import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.stats.api as sms
from statsmodels.stats.power import NormalIndPower
import matplotlib.pyplot as plt
import seaborn as sns

# 中文字体配置
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 加载数据 (2019 年 10 月数据，约 42M 行)
# 数据量很大，先抽样 200 万行加速练习
SAMPLE_SIZE = 2_000_000
df = pd.read_csv('./data/2019-Oct.csv', nrows=SAMPLE_SIZE)
print(f"数据形状: {df.shape}")
print(f"列名: {df.columns.tolist()}")
df.head()

./Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store
License(s): copyright-authors
^C
数据形状: (2000000, 9)
列名: ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


### 2.2 快速 EDA

In [ ]:
# ========== 你的代码 ==========
# 1. 查看 event_type 的分布 (view / cart / purchase 各有多少？)
# 2. 查看 user_id 的唯一用户数
# 3. 计算整体购买转化率 = 有 purchase 行为的用户数 / 总唯一用户数

# 提示:
# df['event_type'].value_counts()
# df['user_id'].nunique()
# df[df['event_type'] == 'purchase']['user_id'].nunique() / df['user_id'].nunique()


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import platform

# 1. 忽略烦人的警告
warnings.filterwarnings('ignore')

# 2. Pandas 显示设置 (破解行列显示限制)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.3f' % x) # 避免科学计数法

# 2. 绘图设置 (高清 + 样式)
%matplotlib inline
sns.set(style='whitegrid', palette='muted', font_scale=1.5)

# 3. 解决中文乱码 (Mac/Windows 自适应) 🇨🇳
if platform.system() == 'Darwin':
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS'] # Mac专用
else:
    plt.rcParams['font.sans-serif'] = ['SimHei'] # Windows专用

plt.rcParams['axes.unicode_minus'] = False # 解决负号显示问题

In [6]:
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.790,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.200,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.100,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.740,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.980,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


In [30]:
df['category_lv1'] = df['category_code'].str.split('.').str[0].fillna('unknown')



In [7]:
# 类型转换
cols_to_str = ['user_id','product_id','category_id']
df[cols_to_str] = df[cols_to_str].astype(str)

In [31]:
df.isnull().mean()
df[df['brand'].isnull() == True]
# 结论:删除brand为null值的行
df = df.dropna(subset='brand')
df.isnull().mean()


event_time      0.000
event_type      0.000
product_id      0.000
category_id     0.000
category_code   0.262
brand           0.000
price           0.000
user_id         0.000
user_session    0.000
category_lv1    0.000
dtype: float64

In [25]:
df['event_type'].unique()

<StringArray>
['view', 'purchase', 'cart']
Length: 3, dtype: str

In [32]:
df.describe(include='all')

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,category_lv1
count,1702478,1702478,1702478,1702478,1255668,1702478,1702478.000,1702478,1702478,1702478
unique,128363,3,60715,526,122,2375,NaN,275406,409927,14
top,2019-10-01 08:42:43 UTC,view,1004856,2053013555631882655,electronics.smartphone,samsung,NaN,555462920,fb075266-182d-4c11-b5f7-4e4dcdabd4a7,electronics
freq,66,1642299,23548,550481,550481,242541,NaN,457,396,757056
mean,NaN,NaN,NaN,NaN,NaN,NaN,314.060,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,381.145,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,0.830,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,70.270,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,174.300,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,386.080,NaN,NaN,NaN


In [37]:
# 用户表构建准备工作
df_user=[]
target1 = ['category_lv1','brand_id','product_id','user_session'] # 用来求对应user_is*event_type下的count/nunique
target2 = ['price'] # 用来求对应的target1下的金额
for cols in target1:
    df_user[f'{cols}_count'] = df.groupby(['user_id','event_type'])[cols].count()

df_user.head()

TypeError: list indices must be integers or slices, not str

In [26]:
df.groupby('event_type')['price'].sum()



event_type
cart        10665695.980
purchase    10495875.250
view       513518277.610
Name: price, dtype: float64

### 2.3 模拟 A/B 分组

> 💡 **为什么要模拟？** 这个数据集本身不包含 A/B 分组字段。在实际工作中，分流由实验平台（如 Google Optimize / 自建平台）自动完成。这里我们用 **Hash 分流** 模拟，确保同一个用户始终在同一组。

**关键技巧**：用 `hash(user_id) % 2` 实现确定性分组，不使用 `random`（否则每次运行分组不同）。

In [ ]:
# ========== 你的代码 ==========
# Task: 构建用户级分析表
#
# Step 1: 创建用户级表 (每行一个用户)
#   - user_id
#   - has_purchase: 该用户是否有购买行为 (0/1)
#   - view_count: 该用户的浏览次数
#   - has_cart: 该用户是否有加购行为 (0/1)
#
# Step 2: 分组 (Hash 分流)
#   - group: 'control' if hash(str(user_id)) % 2 == 0 else 'treatment'
#
# Step 3: 模拟实验效果
#   - 实验组 (treatment) 的转化率人为提升 15% (乘以 1.15 的概率翻转)
#   - 这一步模拟新算法确实有效果的情况
#
# 提示:
# purchase_users = set(df[df['event_type']=='purchase']['user_id'].unique())
# cart_users = set(df[df['event_type']=='cart']['user_id'].unique())
# all_users = df['user_id'].unique()


---

## 3. 分级挑战 🏆

### 🥉 Level 1: 基础 — SRM 检测 + 样本量计算

#### Task 1: SRM (Sample Ratio Mismatch) 检测

> **面试必考**: "你做 A/B 实验会先检查什么？" → SRM！
> 
> 如果实验组和对照组的样本量比例和预期（50:50）不一致，说明分流有 Bug，后续结果全部不可信。

**要求**:
1. 计算 control / treatment 各有多少用户
2. 用 `stats.chisquare()` 检验实际分布是否符合 50:50 期望
3. 输出 P 值，判断 SRM 是否通过（P > 0.05 → 通过）

In [ ]:
# ========== Task 1: SRM 检测 ==========
# 你的代码:



#### Task 2: 样本量计算 (MDE)

> **面试场景**: "PM 说效果至少提升 1 个百分点，你需要多少样本？"

**要求**:
1. 设定基线转化率 p1 = 你在 EDA 中算出的转化率
2. 设定目标转化率 p2 = p1 + 0.01 (提升 1pp)
3. 用 `sms.proportion_effectsize()` 算 Cohen's h
4. 用 `NormalIndPower().solve_power()` 算每组需要的样本量
5. 对比你当前的实际样本量，判断样本是否足够

In [ ]:
# ========== Task 2: 样本量计算 ==========
# 你的代码:



### 🥈 Level 2: 业务 — 显著性检验 + 置信区间

#### Task 3: 显著性检验 (T-Test)

> **核心检验**: 两组 has_purchase (0/1) 的均值差异是否显著。

**要求**:
1. 分别计算两组的 CVR (转化率)
2. 用 `stats.ttest_ind()` 检验差异
3. 输出: P 值、两组 CVR、提升率 (Lift)
4. 给出结论: 显著 / 不显著

In [ ]:
# ========== Task 3: 显著性检验 ==========
# 你的代码:



#### Task 4: 置信区间 (CI)

> **面试加分**: "你只看了 P 值？CI 呢？效果量呢？"

**要求**:
1. 用 `sms.CompareMeans` 计算两组 CVR 差异的 **95% 置信区间**
2. 判断: CI 是否包含 0？(不包含 0 → 与 P < 0.05 等价)
3. 解读: "我们有 95% 的信心认为，新算法的 CVR 提升在 [下界, 上界] 之间"

In [ ]:
# ========== Task 4: 置信区间 ==========
# 你的代码:



#### Task 5: 结果可视化

**要求**:
1. 画一个 **条形图**，对比两组 CVR
2. 添加 **误差线** (Error Bar) 表示置信区间
3. 添加标注: P 值和提升率
4. 图表必须有标题、轴标签、图例

In [ ]:
# ========== Task 5: 结果可视化 ==========
# 你的代码:



### 🥇 Level 3: 进阶 — 分层分析 + 业务决策

#### Task 6: 分层分析 (Segment Analysis)

> **面试杀手锏**: 辛普森悖论! 总体 P < 0.05 不代表每个细分市场都有效。
> 
> "分层分析"是 Senior DA 和 Junior DA 的分水岭。

**要求**:
1. 按 `category_code` 的一级类目拆分（提取 category_code 的第一个 `.` 前的部分作为大类）
2. 对 **Top 5 类目** 分别计算两组 CVR + P 值
3. 输出: 各类目的效果对比表
4. 回答: 是否存在类目表现不一致的情况（提升 vs 下降）？

In [ ]:
# ========== Task 6: 分层分析 ==========
# 你的代码:



#### Task 7: 业务决策报告 (Go / No-Go)

> **面试必考**: "实验显著了，你会直接全量上线吗？"

**要求**:
请用以下模板写出你的决策报告 (Markdown):

```
## 实验结论

### 决策: Go / No-Go / 延长实验

### 主指标
- Control CVR: ___%
- Treatment CVR: ___%
- 提升 (Lift): ___%
- P 值: ___
- 95% CI: [___, ___]

### 护栏指标
- 人均浏览量: Control ___ vs Treatment ___ (P = ___)
- 加购率: Control ___ vs Treatment ___ (P = ___)

### 分层发现
- 最受益类目: ___
- 受损类目 (如有): ___

### 建议
- ...
```

In [ ]:
# ========== Task 7: 护栏指标检验 ==========
# 在写决策报告前，先检查护栏指标是否有异常

# 1. 人均浏览量 (view_count) — 两组是否有显著差异？
# 2. 加购率 (has_cart) — 两组是否有显著差异？
# 你的代码:



##### ✍️ 你的决策报告 (在下面的 Cell 中用 Markdown 撰写)

<!-- 在这里写你的决策报告 -->

## 实验结论

### 决策: 

_（请填写）_

---

## 4. 参考答案 📝

### 参考答案: 数据准备 (EDA + 分组)

In [ ]:
# ====== 参考答案: EDA ======
print("=== 事件分布 ===")
print(df['event_type'].value_counts())
print(f"\n总唯一用户数: {df['user_id'].nunique():,}")

# 整体购买转化率
purchase_users = df[df['event_type'] == 'purchase']['user_id'].nunique()
total_users = df['user_id'].nunique()
baseline_cvr = purchase_users / total_users
print(f"基线购买转化率: {baseline_cvr:.4f} ({baseline_cvr*100:.2f}%)")

In [ ]:
# ====== 参考答案: 构建用户级分析表 + 模拟 A/B 分组 ======
import hashlib

# Step 1: 用户级聚合
purchase_user_set = set(df[df['event_type'] == 'purchase']['user_id'].unique())
cart_user_set = set(df[df['event_type'] == 'cart']['user_id'].unique())

user_views = df[df['event_type'] == 'view'].groupby('user_id').size().reset_index(name='view_count')
all_user_ids = df['user_id'].unique()

user_df = pd.DataFrame({'user_id': all_user_ids})
user_df['has_purchase'] = user_df['user_id'].isin(purchase_user_set).astype(int)
user_df['has_cart'] = user_df['user_id'].isin(cart_user_set).astype(int)
user_df = user_df.merge(user_views, on='user_id', how='left')
user_df['view_count'] = user_df['view_count'].fillna(0).astype(int)

# Step 2: Hash 分流 (确定性分组)
def assign_group(uid: int) -> str:
    """基于 user_id 的 Hash 值进行确定性 A/B 分组"""
    hash_val = int(hashlib.md5(str(uid).encode()).hexdigest(), 16)
    return 'control' if hash_val % 2 == 0 else 'treatment'

user_df['group'] = user_df['user_id'].apply(assign_group)

# Step 3: 模拟实验效果 (Treatment 组额外提升 15% 转化)
EFFECT_MULTIPLIER = 1.15  # 新算法让转化概率提升 15%
np.random.seed(42)

treatment_mask = user_df['group'] == 'treatment'
non_purchase_treatment = treatment_mask & (user_df['has_purchase'] == 0)

# 对 treatment 中未购买的用户，按 baseline_cvr * (EFFECT_MULTIPLIER - 1) 的概率翻转为购买
flip_prob = baseline_cvr * (EFFECT_MULTIPLIER - 1)
random_flip = np.random.random(non_purchase_treatment.sum()) < flip_prob
user_df.loc[non_purchase_treatment, 'has_purchase'] = random_flip.astype(int) | user_df.loc[non_purchase_treatment, 'has_purchase'].values

print(f"用户级表形状: {user_df.shape}")
print(f"分组分布:\n{user_df['group'].value_counts()}")
print(f"\nControl CVR: {user_df[user_df['group']=='control']['has_purchase'].mean():.4f}")
print(f"Treatment CVR: {user_df[user_df['group']=='treatment']['has_purchase'].mean():.4f}")

### 参考答案: Level 1 (SRM + 样本量)

In [ ]:
# ====== 参考答案: Task 1 — SRM 检测 ======
group_counts = user_df['group'].value_counts()
n_ctrl = group_counts.get('control', 0)
n_test = group_counts.get('treatment', 0)
n_total = n_ctrl + n_test

# 期望: 50:50 分配
expected_each = n_total / 2

# 卡方拟合优度检验
chi2_stat, srm_p = stats.chisquare([n_ctrl, n_test], [expected_each, expected_each])

print(f"Control: {n_ctrl:,} | Treatment: {n_test:,}")
print(f"期望各: {expected_each:,.0f}")
print(f"SRM 检验: χ² = {chi2_stat:.4f}, P = {srm_p:.4f}")
print(f"SRM 结论: {'✅ 通过 (P > 0.05)' if srm_p > 0.05 else '❌ 失败 (P < 0.05)'}")  

In [ ]:
# ====== 参考答案: Task 2 — 样本量计算 ======
p1 = baseline_cvr  # 基线转化率
p2 = p1 + 0.01     # MDE: 提升 1 个百分点

# Cohen's h 效应量
effect_size = sms.proportion_effectsize(p1, p2)

# 求解每组所需样本量
required_n = NormalIndPower().solve_power(
    effect_size=effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1  # 1:1 分组
)

print(f"基线 CVR: {p1:.4f} | 目标 CVR: {p2:.4f}")
print(f"Cohen's h: {effect_size:.4f}")
print(f"每组需要样本: {int(required_n):,}")
print(f"总共需要样本: {int(required_n) * 2:,}")
print(f"\n当前实际样本: Control={n_ctrl:,}, Treatment={n_test:,}")
print(f"样本是否足够: {'✅ 足够' if min(n_ctrl, n_test) >= required_n else '❌ 不足'}")

### 参考答案: Level 2 (检验 + CI + 可视化)

In [ ]:
# ====== 参考答案: Task 3 — 显著性检验 ======
ctrl = user_df[user_df['group'] == 'control']['has_purchase']
test = user_df[user_df['group'] == 'treatment']['has_purchase']

cvr_ctrl = ctrl.mean()
cvr_test = test.mean()
lift = (cvr_test - cvr_ctrl) / cvr_ctrl

t_stat, p_value = stats.ttest_ind(test, ctrl)

print(f"Control CVR: {cvr_ctrl:.4f} ({cvr_ctrl*100:.2f}%)")
print(f"Treatment CVR: {cvr_test:.4f} ({cvr_test*100:.2f}%)")
print(f"Lift: {lift:+.2%}")
print(f"T-stat: {t_stat:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"\n结论: {'✅ 显著 (P < 0.05) — 新算法有效!' if p_value < 0.05 else '❌ 不显著 (P >= 0.05) — 无法证明差异'}")

In [ ]:
# ====== 参考答案: Task 4 — 置信区间 ======
cm = sms.CompareMeans(
    sms.DescrStatsW(test.values),
    sms.DescrStatsW(ctrl.values)
)
ci_low, ci_high = cm.tconfint_diff(usevar='unequal', alpha=0.05)

print(f"CVR 差异的 95% 置信区间: [{ci_low:.4f}, {ci_high:.4f}]")
print(f"差异绝对值: {cvr_test - cvr_ctrl:.4f}")
print(f"\nCI 是否包含 0: {'❌ 否 → 显著' if (ci_low > 0 or ci_high < 0) else '⚠️ 是 → 不显著'}")
print(f"\n面试话术: '我们有 95% 的信心认为，新推荐算法使购买转化率提升了 {ci_low*100:.2f}% ~ {ci_high*100:.2f}%。'")

In [ ]:
# ====== 参考答案: Task 5 — 结果可视化 ======
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 图 1: 两组 CVR 对比 + 误差线
groups = ['Control', 'Treatment']
cvrs = [cvr_ctrl, cvr_test]
# 比例的标准误: SE = sqrt(p*(1-p)/n)
se_ctrl = np.sqrt(cvr_ctrl * (1 - cvr_ctrl) / len(ctrl))
se_test = np.sqrt(cvr_test * (1 - cvr_test) / len(test))
errors = [1.96 * se_ctrl, 1.96 * se_test]

bars = axes[0].bar(groups, cvrs, yerr=errors, capsize=8,
                   color=['#5B9BD5', '#ED7D31'], alpha=0.85, edgecolor='white', linewidth=1.5)
axes[0].set_ylabel('购买转化率 (CVR)')
axes[0].set_title(f'A/B 实验结果: CVR 对比\nP={p_value:.4f} | Lift={lift:+.2%}')
for bar, cvr in zip(bars, cvrs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{cvr:.4f}', ha='center', va='bottom', fontweight='bold')

# 图 2: CVR 差异的置信区间
diff = cvr_test - cvr_ctrl
axes[1].errorbar(x=0, y=diff, yerr=[[diff - ci_low], [ci_high - diff]],
                fmt='o', color='#ED7D31', markersize=10, capsize=12, capthick=2)
axes[1].axhline(y=0, color='gray', linestyle='--', linewidth=1)
axes[1].set_xlim(-0.5, 0.5)
axes[1].set_xticks([])
axes[1].set_ylabel('CVR 差异 (Treatment - Control)')
axes[1].set_title(f'95% 置信区间\n[{ci_low:.4f}, {ci_high:.4f}]')

plt.tight_layout()
plt.savefig('ab_test_result.png', dpi=150, bbox_inches='tight')
plt.show()

### 参考答案: Level 3 (分层分析 + 护栏)

In [ ]:
# ====== 参考答案: Task 6 — 分层分析 ======
# 获取用户最常浏览的一级类目
df_with_cat = df[df['category_code'].notna()].copy()
df_with_cat['main_category'] = df_with_cat['category_code'].str.split('.').str[0]

# 每个用户取最常浏览的类目
user_top_cat = (
    df_with_cat.groupby(['user_id', 'main_category'])
    .size()
    .reset_index(name='cnt')
    .sort_values('cnt', ascending=False)
    .drop_duplicates(subset='user_id', keep='first')
    [['user_id', 'main_category']]
)

# 合并到用户表
user_seg = user_df.merge(user_top_cat, on='user_id', how='left')
user_seg['main_category'] = user_seg['main_category'].fillna('unknown')

# Top 5 类目分层检验
top5_cats = user_seg['main_category'].value_counts().head(5).index.tolist()

print("=" * 70)
print(f"{'类目':<15} {'Ctrl CVR':>10} {'Test CVR':>10} {'Lift':>8} {'P-value':>10} {'判断':>6}")
print("=" * 70)

for cat in top5_cats:
    seg = user_seg[user_seg['main_category'] == cat]
    seg_ctrl = seg[seg['group'] == 'control']['has_purchase']
    seg_test = seg[seg['group'] == 'treatment']['has_purchase']
    
    seg_cvr_ctrl = seg_ctrl.mean()
    seg_cvr_test = seg_test.mean()
    seg_lift = (seg_cvr_test - seg_cvr_ctrl) / seg_cvr_ctrl if seg_cvr_ctrl > 0 else 0
    _, seg_p = stats.ttest_ind(seg_test, seg_ctrl)
    sig = '✅' if seg_p < 0.05 else '—'
    
    print(f"{cat:<15} {seg_cvr_ctrl:>10.4f} {seg_cvr_test:>10.4f} {seg_lift:>+7.2%} {seg_p:>10.4f} {sig:>6}")

print("=" * 70)

In [ ]:
# ====== 参考答案: Task 7 — 护栏指标检验 ======
# 1. 人均浏览量
view_ctrl = user_df[user_df['group'] == 'control']['view_count']
view_test = user_df[user_df['group'] == 'treatment']['view_count']
_, view_p = stats.ttest_ind(view_test, view_ctrl)

print("=== 护栏指标 ===")
print(f"人均浏览量: Control={view_ctrl.mean():.2f} vs Treatment={view_test.mean():.2f} (P={view_p:.4f})")

# 2. 加购率
cart_ctrl = user_df[user_df['group'] == 'control']['has_cart']
cart_test = user_df[user_df['group'] == 'treatment']['has_cart']
_, cart_p = stats.ttest_ind(cart_test, cart_ctrl)

print(f"加购率: Control={cart_ctrl.mean():.4f} vs Treatment={cart_test.mean():.4f} (P={cart_p:.4f})")
print(f"\n护栏结论: {'✅ 护栏指标均无异常' if view_p > 0.05 and cart_p > 0.05 else '⚠️ 護欄指标有变化，需要关注'}")

---

## 5. 结论与建议 (Actionable Insights) 📊

### 面试话术模板

**开头** (20秒):
> "我们对新推荐算法进行了 A/B 测试，总共覆盖了 XX 万用户，实验周期为 X 天。"

**结果** (30秒):
> "实验组的购买转化率为 X%，对比对照组的 X% 提升了 X%，P 值为 X，95% 置信区间为 [X, X]，效果在统计上显著。"

**分层** (20秒):
> "分层分析发现，效果在电子品类最为显著（提升 X%），但在服装品类无显著差异，建议后续针对性优化。"

**建议** (20秒):
> "建议全量上线新算法，但需持续监控服装品类的表现，并在下个迭代中重点优化该品类的推荐逻辑。"

### 关键检查清单 (面试前必看)

- [ ] SRM 检测通过？
- [ ] 样本量足够？
- [ ] P < 0.05 且 CI 不跨 0？
- [ ] 护栏指标无恶化？
- [ ] 分层分析无辛普森悖论？
- [ ] 效果有业务意义（不只是统计显著）？